In [1]:
!pip install -r ../requirements.txt 

  Using cached plyfile-1.1.2-py3-none-any.whl.metadata (43 kB)
  Using cached onnxruntime_gpu-1.22.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.1 kB)
  Using cached coloredlogs-15.0.1-py2.py3-none-any.whl.metadata (12 kB)
  Using cached humanfriendly-10.0-py2.py3-none-any.whl.metadata (9.2 kB)
Using cached plyfile-1.1.2-py3-none-any.whl (36 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.2/283.2 MB 384.4 kB/s eta 0:00:0000:0100:16


In [9]:
import mlflow.pyfunc
import base64
import subprocess
from pathlib import Path
from mlflow.types import Schema, ColSpec
from mlflow.models import ModelSignature
import os
import shutil
import gc
import numpy as np
from PIL import Image
from io import BytesIO
from plyfile import PlyData
import torch
import onnxruntime as ort
from torchvision import transforms

class Mira3DPipeline(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        base_path = Path(context.artifacts["colmap_bundle"])
        self.ffmpeg_bin = base_path / "ffmpeg" / "bin" / "ffmpeg"
        self.magick_bin = base_path / "imagemagick" / "bin" / "magick"
        self.colmap_bin = base_path / "colmap" / "bin" / "colmap"
        self.brush_bin = base_path / "brush" / "brush_app"

        self.onnx_model_path = Path(context.artifacts["onnx_model"])

        self.env = os.environ.copy()
        self.env["LD_LIBRARY_PATH"] = (
            f"{base_path / 'ffmpeg' / 'lib'}:"
            f"{base_path / 'imagemagick' / 'lib'}:"
            f"{base_path / 'colmap' / 'lib'}:"
            f"{self.env.get('LD_LIBRARY_PATH', '')}"
        )

        self.onnx_session = ort.InferenceSession(str(self.onnx_model_path), providers=["CUDAExecutionProvider"])
        self.input_name = self.onnx_session.get_inputs()[0].name
        self.onnx_transform = transforms.Compose([
            transforms.Resize((1024, 1024)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def decode_and_save_video(self, base64_str: str, save_path: Path):
        video_bytes = base64.b64decode(base64_str)
        with open(save_path, "wb") as f:
            f.write(video_bytes)

    def extract_frames(self, input_path: Path, output_dir: Path, fps="2"):
        output_dir.mkdir(parents=True, exist_ok=True)
        subprocess.run([
            str(self.ffmpeg_bin), "-hwaccel", "cuda", "-i", str(input_path),
            "-vf", f"fps={fps}", str(output_dir / "%04d.jpg")
        ], env=self.env, check=True)

    def downsample_images(self, src_dir: Path, dest_dir: Path):
        dest_dir.mkdir(exist_ok=True)
        subprocess.run(
            f"{str(self.magick_bin)} mogrify -path {dest_dir} -resize 50% {src_dir}/*.jpg",
            shell=True, check=True, env=self.env
        )

    def remove_background(self, input_dir: Path, output_dir: Path):
        output_dir.mkdir(exist_ok=True)

        def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -250, 250)))

        for img_path in input_dir.glob("*.jpg"):
            img = Image.open(img_path).convert("RGB")
            w, h = img.size
            tensor = self.onnx_transform(img).unsqueeze(0).numpy().astype(np.float32)
            pred = self.onnx_session.run(None, {self.input_name: tensor})[0].squeeze()
            mask = Image.fromarray((sigmoid(pred) * 255).astype(np.uint8)).resize((w, h))
            mask_arr = np.array(mask) / 255.0
            result_img = img.copy()
            result_img.putalpha(Image.fromarray((mask_arr * 255).astype(np.uint8)))
            result_img.save(output_dir / f"{img_path.stem}.png")

        torch.cuda.empty_cache()
        gc.collect()

    def run_colmap_and_brush(self, frames_dir: Path, output_dir: Path) -> Path:
        db_path = output_dir / "database.db"
        sparse_dir = output_dir / "sparse"
        sparse_dir.mkdir(parents=True, exist_ok=True)

        subprocess.run([
            str(self.colmap_bin), "feature_extractor",
            "--database_path", str(db_path),
            "--image_path", str(frames_dir),
            "--ImageReader.single_camera", "1",
            "--ImageReader.camera_model", "PINHOLE",
            "--SiftExtraction.use_gpu", "1"
        ], env=self.env, check=True)

        subprocess.run([
            str(self.colmap_bin), "exhaustive_matcher",
            "--database_path", str(db_path),
            "--SiftMatching.use_gpu", "1"
        ], env=self.env, check=True)

        subprocess.run([
            str(self.colmap_bin), "mapper",
            "--database_path", str(db_path),
            "--image_path", str(frames_dir),
            "--output_path", str(sparse_dir)
        ], env=self.env, check=True)

        model_dir = sparse_dir / "0"
        brush_dir = output_dir / "brush"
        brush_dir.mkdir(parents=True, exist_ok=True)
        ply_name = "output.ply"

        subprocess.run([
            str(self.brush_bin), str(model_dir),
            "--total-steps", "30000",
            "--export-path", str(brush_dir),
            "--export-name", ply_name,
            "--export-every", "30000"
        ], env=self.env, check=True)

        return brush_dir / ply_name

    def convert_ply_to_splat(self, ply_path: Path) -> Path:
        plydata = PlyData.read(ply_path)
        vert = plydata["vertex"]

        sorted_indices = np.argsort(
            -np.exp(vert["scale_0"] + vert["scale_1"] + vert["scale_2"])
            / (1 + np.exp(-vert["opacity"]))
        )

        buffer = BytesIO()
        for idx in sorted_indices:
            v = vert[idx]
            position = np.array([v["x"], v["y"], v["z"]], dtype=np.float32)
            scales = np.exp(np.array([v["scale_0"], v["scale_1"], v["scale_2"]], dtype=np.float32))
            rot = np.array([v["rot_0"], v["rot_1"], v["rot_2"], v["rot_3"]], dtype=np.float32)
            SH_C0 = 0.28209479177387814
            color = np.array([
                0.5 + SH_C0 * v["f_dc_0"],
                0.5 + SH_C0 * v["f_dc_1"],
                0.5 + SH_C0 * v["f_dc_2"],
                1 / (1 + np.exp(-v["opacity"])),
            ])
            buffer.write(position.tobytes())
            buffer.write(scales.tobytes())
            buffer.write((color * 255).clip(0, 255).astype(np.uint8).tobytes())
            buffer.write(((rot / np.linalg.norm(rot)) * 128 + 128).clip(0, 255).astype(np.uint8).tobytes())

        splat_path = ply_path.with_suffix(".splat")
        with open(splat_path, "wb") as f:
            f.write(buffer.getvalue())
        return splat_path

    def predict(self, context, model_input):
        video_b64 = model_input["video_base64"][0]
        use_downsample = model_input["use_downsample"][0]

        run_dir = Path("/tmp/mira3d")
        input_video_path = run_dir / "input.mp4"
        frames_dir = run_dir / "frames"
        masked_dir = run_dir / "masked"
        output_dir = run_dir / "colmap"

        shutil.rmtree(run_dir, ignore_errors=True)
        run_dir.mkdir(parents=True)

        self.decode_and_save_video(video_b64, input_video_path)
        self.extract_frames(input_video_path, frames_dir)

        if use_downsample:
            downsample_dir = run_dir / "frames_ds"
            self.downsample_images(frames_dir, downsample_dir)
            frames_dir = downsample_dir

        self.remove_background(frames_dir, masked_dir)
        splat_path = self.convert_ply_to_splat(self.run_colmap_and_brush(masked_dir, output_dir))

        return [{
            "status": "Completed",
            "splat_file": str(splat_path),
            "size_bytes": splat_path.stat().st_size
        }]

# Define input/output schema
input_schema = Schema([
    ColSpec("string", "video_base64"),
    ColSpec("boolean", "use_downsample")
])
output_schema = Schema([
    ColSpec("string", "status"),
    ColSpec("string", "splat_file"),
    ColSpec("integer", "size_bytes")
])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)


In [8]:
import mlflow

mlflow.set_tracking_uri("/phoenix/mlflow")
mlflow.set_experiment("Mira3D_Pipeline")

with mlflow.start_run(run_name="mira_model") as run:
    mlflow.pyfunc.log_model(
        artifact_path="mira3d_pipeline_v3",
        python_model=Mira3DPipeline(),
        signature=signature,
        artifacts={
            "colmap_bundle": "../export", 
            "demo": "../frontend/dist",
            "onnx_model": "../../datafabric/birefnetgeneral-model/birefnet-general.onnx",
        },
        pip_requirements="../requirements.txt"
    )

    mlflow.register_model(
        model_uri=f"runs:/{run.info.run_id}/mira3d_pipeline_v3",
        name="mira3d_pipeline_v3"
    )

print("model registered.")


model registered.


Registered model 'mira3d_pipeline_v3' already exists. Creating a new version of this model...
Created version '6' of model 'mira3d_pipeline_v3'.
